### GOLD TESTING - DIMENSION VOUCHERS (SCD2)

#### Purpose
- Validate `coffee.gold.dim_vouchers` against `coffee.silver.vouchers`
- Ensure SCD Type 2 logic is working correctly
- Verify current records using `__END_AT IS NULL`
- Validate referential integrity from `fact_transactions` to `dim_vouchers`

#### Tests Covered
1. Silver vs Gold row count reconciliation (current rows only)
2. Null checks on business key (`voucher_id`)
3. SCD2 sanity check: only 1 current row per `voucher_id`
4. SCD2 sanity check: invalid date ranges (`__END_AT < __START_AT`)
5. Referential integrity: `fact_transactions.voucher_id` must exist in current `dim_vouchers`
   - Only validated for non-null voucher_id values


In [0]:
-- TEST 1: SILVER vs GOLD CURRENT ROW COUNT
SELECT
  'dim_vouchers_count_recon' AS test_name,
  (SELECT COUNT(*) FROM coffee.silver.vouchers) AS silver_count,
  (SELECT COUNT(*) FROM coffee.gold.dim_vouchers WHERE __END_AT IS NULL) AS gold_current_count;

In [0]:
-- TEST 2: NULL CHECK ON BUSINESS KEY
SELECT
  'dim_vouchers_null_voucher_id' AS test_name,
  COUNT(*) AS null_key_count
FROM coffee.gold.dim_vouchers
WHERE voucher_id IS NULL;

In [0]:
-- TEST 3: SCD2 SANITY - ONLY 1 CURRENT ROW PER VOUCHER
SELECT
  'dim_vouchers_multiple_current_rows' AS test_name,
  COUNT(*) AS keys_with_multiple_current
FROM (
  SELECT voucher_id
  FROM coffee.gold.dim_vouchers
  WHERE __END_AT IS NULL
  GROUP BY voucher_id
  HAVING COUNT(*) > 1
);

In [0]:
-- TEST 4: SCD2 SANITY - INVALID DATE RANGE CHECK
SELECT
  'dim_vouchers_invalid_date_ranges' AS test_name,
  COUNT(*) AS invalid_rows
FROM coffee.gold.dim_vouchers
WHERE __END_AT IS NOT NULL AND __END_AT < __START_AT;

In [0]:
-- TEST 5: REFERENTIAL INTEGRITY - FACT TRANSACTIONS -> DIM VOUCHERS
-- Note:
--   voucher_id can be NULL in fact table, so only validating non-null values
SELECT
  'RI_fact_transactions_voucher_id' AS test_name,
  COUNT(*) AS missing_fk_count
FROM coffee.gold.fact_transactions f
LEFT JOIN coffee.gold.dim_vouchers d
  ON f.voucher_id = d.voucher_id AND d.__END_AT IS NULL
WHERE f.voucher_id IS NOT NULL
  AND d.voucher_id IS NULL;